## **RDD Persistence and Caching**

In [1]:
# PySpark 설정
from pyspark import SparkConf, SparkContext

conf = SparkConf().setMaster('local').setAppName('cache-and-persist')
sc = SparkContext(conf=conf)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/29 04:10:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
sc

<SparkContext master=local appName=cache-and-persist>

#### **RDD에 대한 Partitioning 수행**
Partitioning과 Shuffling은 어떻게 일어나는지 확인하기

In [3]:
pairs = sc.parallelize([1,2,3,4,2,4,1]).map(lambda x : (x, x))
pairs.collect()

[(1, 1), (2, 2), (3, 3), (4, 4), (2, 2), (4, 4), (1, 1)]

In [4]:
pairs.partitionBy(2).glom().collect()

[[(2, 2), (4, 4), (2, 2), (4, 4)], [(1, 1), (3, 3), (1, 1)]]

In [5]:
pairs.partitionBy(2, lambda x: x % 2).glom().collect()

[[(2, 2), (4, 4), (2, 2), (4, 4)], [(1, 1), (3, 3), (1, 1)]]

# **persist( )을 활용한 성능 개선**

In [6]:
path = "../data/"
filename = "fhvhv_tripdata_2020-03_short.csv"

In [7]:
lines = sc.textFile(path + filename)   # .csv -> RDD object

In [8]:
header = lines.first()

In [9]:
header

'hvfhs_license_num,dispatching_base_num,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,SR_Flag'

In [10]:
# 헤더 제거: filter(), 변환
filtered_lines = lines.filter(lambda row:row != header)

In [11]:
# RDD 값 1개 출력: top(), 액션
result1 = filtered_lines.top(1)

In [12]:
# RDD 값 전체 출력: collect(), 액션
%time result2 = filtered_lines.collect()  # filter() 변환 중복 실행

CPU times: user 381 ms, sys: 511 ms, total: 892 ms
Wall time: 6.49 s


In [13]:
# persist() 
filtered_lines = lines.filter(lambda row:row != header).persist()

In [14]:
result1 = filtered_lines.top(1)

In [15]:
%time result2 = filtered_lines.collect()

CPU times: user 599 ms, sys: 486 ms, total: 1.08 s
Wall time: 2.11 s
